# 02 — Preprocessing

Splits the input sample into train/test sets, scales numeric columns, and produces a SMOTE-balanced variant of the training set for the imbalance-handling comparison. Requires `01_eda.ipynb` (and, if `config.USE_ENGINEERED_FEATURES` is `True`, also `01b_feature_engineering.ipynb`) to have been run first.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

from config import (
    DATA_DIR, SAMPLED_CSV, SAMPLED_ENGINEERED_CSV, USE_ENGINEERED_FEATURES,
    TARGET_COL, TEST_SIZE, RANDOM_STATE,
)

BASE_NUMERIC_COLS = ["BMI", "MentHlth", "PhysHlth"]
# Engineered numeric columns that get StandardScaler treatment too, if present
# (only relevant when USE_ENGINEERED_FEATURES = True and the feature survived
# 01b_feature_engineering.ipynb's ablation check).
OPTIONAL_ENGINEERED_NUMERIC_COLS = ["RiskScore", "TotalUnhealthyDays"]

## Load the input sample

Controlled by `config.USE_ENGINEERED_FEATURES` (default `True`): reads the output of `01b_feature_engineering.ipynb` if enabled, otherwise falls back to the untouched `01_eda.ipynb` sample with all 21 raw features and no engineered columns.

In [2]:
input_csv = SAMPLED_ENGINEERED_CSV if USE_ENGINEERED_FEATURES else SAMPLED_CSV

if not input_csv.exists():
    hint = (
        "Run 01b_feature_engineering.ipynb first, or set USE_ENGINEERED_FEATURES = False "
        "in config.py to use the raw sample instead."
        if USE_ENGINEERED_FEATURES
        else "Run 01_eda.ipynb first to generate it."
    )
    raise FileNotFoundError(f"Could not find {input_csv}. {hint}")

df = pd.read_csv(input_csv)
print(f"Loaded {input_csv.name} — shape: {df.shape}")
print(f"USE_ENGINEERED_FEATURES = {USE_ENGINEERED_FEATURES}")
df.head()

Loaded diabetes_sampled_final.csv — shape: (40000, 26)
USE_ENGINEERED_FEATURES = True


,Diabetes_binary,HighBP,HighChol,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,Sex,Age,Education,Income,BMI_Underweight,BMI_Normal,BMI_Slightly_Overweight,BMI_Overweight,BMI_Obese,RiskScore
0,0.0,0.0,0.0,32.0,1.0,0.0,0.0,0.0,1.0,1.0,...,0.0,7.0,4.0,3.0,False,False,False,True,False,1.0
1,0.0,1.0,0.0,27.0,0.0,0.0,0.0,1.0,1.0,1.0,...,1.0,10.0,6.0,8.0,False,False,True,False,False,1.0
2,0.0,0.0,0.0,23.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,13.0,6.0,8.0,False,True,False,False,False,1.0
3,1.0,0.0,0.0,26.0,1.0,0.0,0.0,1.0,1.0,1.0,...,0.0,13.0,3.0,2.0,False,False,True,False,False,1.0
4,0.0,0.0,1.0,37.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,11.0,6.0,3.0,False,False,False,True,False,2.0


## Train/test split + scaling

80/20 stratified split, `StandardScaler` fit on the training set only (numeric columns) to avoid data leakage.

In [3]:
NUMERIC_COLS = [c for c in BASE_NUMERIC_COLS + OPTIONAL_ENGINEERED_NUMERIC_COLS if c in df.columns]
print(f"Numeric columns to scale: {NUMERIC_COLS}")

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train = X_train.copy()
X_test = X_test.copy()
X_train[NUMERIC_COLS] = scaler.fit_transform(X_train[NUMERIC_COLS])
X_test[NUMERIC_COLS] = scaler.transform(X_test[NUMERIC_COLS])

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print("Train class balance (original):")
print(y_train.value_counts(normalize=True))

Numeric columns to scale: ['BMI', 'MentHlth', 'PhysHlth', 'RiskScore']
Train shape: (32000, 25), Test shape: (8000, 25)
Train class balance (original):
Diabetes_binary
0.0    0.860688
1.0    0.139313
Name: proportion, dtype: float64


## SMOTE-balanced training variant

In [4]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print("Train class balance (after SMOTE):")
print(y_train_balanced.value_counts(normalize=True))

Train class balance (after SMOTE):
Diabetes_binary
0.0    0.5
1.0    0.5
Name: proportion, dtype: float64


## Save splits for downstream notebooks

In [5]:
X_train.to_csv(DATA_DIR / "X_train.csv", index=False)
X_test.to_csv(DATA_DIR / "X_test.csv", index=False)
y_train.to_csv(DATA_DIR / "y_train.csv", index=False)
y_test.to_csv(DATA_DIR / "y_test.csv", index=False)

X_train_balanced.to_csv(DATA_DIR / "X_train_balanced.csv", index=False)
y_train_balanced.to_csv(DATA_DIR / "y_train_balanced.csv", index=False)

print(f"Saved processed splits to {DATA_DIR}")

Saved processed splits to C:\Users\ardao\Desktop\Projects\Bil476\data
